# Survey Tasks and Actual Attributions

In this notebook, we add a binary column to the `survey_tasks` dataframe (indicating whether the individual performed the task or not) in order to explore a link between aspiration, experience, and skill with task performance later on.

The data comes from the 2023 survey, conducted during the iGEM Grand Jamboree. In the tasks portion of the survey, participants were asked to rate their Certainty, Experience and Preference for each of the 17 possible tasks (not answering if they actually performed the task or not). Since teams started having structured attributions webpages starting from 2023, surveys for previous years are not taken into account here. Additionally, the survey was not conducted in 2024 when teams had structured attribution forms. By connecting survey tasks and attributions data, we can add the `"TaskPerformed"` column, with value 0 if the participant has not performed a certain task (i.e. the task is not listed in their attribution form), or 1 if they have performed it.

The `survey_tasks` dataframe includes the columns: `"team"`, `"participant"`, `"Task"`, `"Certainty"`, `"Experience"`, and `"Preference"`. The `"participant"` is the team member’s full name, hashed for anonymity.

The survey and attributions are connected using the helper 2023 roster dataframe, by merging them on username first, and then fuzzy matching full names.

The resulting dataframe, now with the added `"TaskPerformed"` binary column, is saved as a TSV file in the `results` directory.
Exploratory data analysis is conducted on this file in `notebooks/4_survey_tasks_and_attributions_EDA.ipynb`.


In [1]:
import pandas as pd
import hashlib
from rapidfuzz import process, fuzz

## 1. Loading and Cleaning Survey, Roster, and Attributions Data

In [2]:
survey_tasks_df = pd.read_csv("../data/igem_ties_surveys/2023/survey_tasks.csv")

In [ ]:
survey_tasks_df.head()

,team,participant,Task,Certainty,Experience,Preference
0,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,Analysis,Somewhat unsure,Between one and two years,Neutral
1,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,Background Research,Somewhat sure,Between one and two years,Neutral
2,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,Conceptualization,Neutral,Less than one year,Neutral
3,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,Data Curation,Somewhat unsure,Less than one year,Somewhat disagree
4,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,Entrepreneurship,Neutral,Less than one year,Somewhat disagree


In [3]:
roster_2023_df = pd.read_csv("../data/igem_ties_surveys/2023/iGEM_2023_roster.csv")

In [29]:
# Attributions from the results directory have the Username column as well
attributions_df = pd.read_table("../results/attributions/attributions_2023_2024.tsv")
# look at just 2023
attributions_2023_df = attributions_df[attributions_df["Year"] == 2023].copy()

In [30]:
roster_2023_df["Participant"].nunique()

8597

In [31]:
attributions_2023_df["FullName"].nunique()

7970

Tasks in attributions have dashes and are lowercase (when taken from the attributions-form but fixed in the results, so no need to do the cleaning if reading the file from results). So, we will replace dashes with spaces in attributions tasks, convert survey_tasks to lowercase, and strip leading/trailing whitespaces from both just in case. 

In [5]:
# Clean Task column
 
'''
- Convert Task to lowercase
- Replace dashes with spaces
- Strip leading and trailing spaces
'''

def clean_task_name(df):
    if "Task" in df.columns:
        df["Task"] = df["Task"].astype(str).str.lower().str.replace("-", " ").str.strip()
    return df

In [32]:
attributions_df = clean_task_name(attributions_2023_df)
attributions_df.head()

,FullName,Username,RosterUUID,TeamID,Team,Year,Role,Task,TaskDescription
106,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,conceptualization,Was a part of brainstorming for the project id...
107,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,project administration,Organized and supervised the work that the tea...
108,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,entrepreneurship,Worked on the business analysis models
109,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,public engagement,Planned and organized multiple education visit...
110,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,writing,"Wrote the Education, Collaborations, Implement..."


In [7]:
survey_tasks_df_with_cleaned_task_name = clean_task_name(survey_tasks_df)
survey_tasks_df_with_cleaned_task_name.head()

,team,participant,Task,Certainty,Experience,Preference
0,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,analysis,Somewhat unsure,Between one and two years,Neutral
1,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,background research,Somewhat sure,Between one and two years,Neutral
2,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,conceptualization,Neutral,Less than one year,Neutral
3,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,data curation,Somewhat unsure,Less than one year,Somewhat disagree
4,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,entrepreneurship,Neutral,Less than one year,Somewhat disagree


## 2. Merging Roster and Attributions Data

We are going to merge attributions and roster data based on the username, and add a Participant column to the attributions data (so that it can be later hashed and matched with survey data). Keep in mind that Username is called Email in the roster data, and that both should be lowercase, have dashes removed, stripped of leading and trailing whitespaces before merging.

In [33]:
# Clean Username and Email so they are ready for matching

# attributions_2023_df['OriginalUsername'] = attributions_2023_df['Username'] # if we want to keep the original username for reference
# attributions_2023_df['CleanUsername'] = (...) #and then create a new column with cleaned usernames


attributions_2023_df['Username'] = (
    attributions_2023_df['Username']
    .str.lower() #convert to lowercase
    .str.strip() #remove leading and trailing spaces
    .str.replace('-', '', regex=False)
    .str.replace('_', '', regex=False) #remove dashes and underscores
    .str.replace(' ', '', regex=False) #remove spaces
    .str.replace(r'0+$', '', regex=True)  #remove trailing zeros
    .str.replace(r'^us', '', regex=True)      #remove leading 'us'
    .str.replace(r'^u', '', regex=True) #remove leading 'us'
)

roster_2023_df['Email'] = (
    roster_2023_df['Email']
    .str.lower()
    .str.strip()
    .str.replace('-', '', regex=False)
    .str.replace('_', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(r'0+$', '', regex=True)
    .str.replace(r'^us', '', regex=True)  
    .str.replace(r'^u', '', regex=True)     
)

When looking at unmatched roster_2023 and attributions usernames, we noticed that a lot of the times they have added dashes, trailing zeros, or leading 'us'/'u' in attributions (and attributions fetches 2025 roster from an API so it is the most recent version of data). Also, some of them are capitalized and some are not. In order to match these usernames, we need to add all of these fixes.

In [ ]:
roster_2023_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8831 entries, 0 to 8830
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Team         8831 non-null   object
 1   Participant  8831 non-null   object
 2   Email        8831 non-null   object
dtypes: object(3)
memory usage: 207.1+ KB


In [ ]:
roster_2023_df["Participant"].nunique()

8597

In [ ]:
# Number of unique Emails
roster_2023_df["Email"].nunique()

8674

In [ ]:
# Number of rows in roster
roster_2023_df.shape[0]

8831

In [ ]:
duplicate_emails = roster_2023_df["Email"].value_counts()
duplicate_emails = duplicate_emails[duplicate_emails > 1]

print(duplicate_emails)

Email
professorli    12
slf0905         8
lucyshi2018     8
guanghui        6
xywei           6
               ..
jionghong       2
lovefifa        2
suzansq         2
diolatcg        2
bellaa          2
Name: count, Length: 93, dtype: int64


Eighty emails appear more than once in the roster data, with slightly different Team names. We will merge on team name and username then.

In [34]:
merged_attributions_and_roster_df = attributions_2023_df.merge(
    roster_2023_df[['Team', 'Participant', 'Email']],
    how='left',
    left_on=['Team', 'Username'],
    right_on=['Team', 'Email']
)

# Drop redundant 'Email' column 
merged_attributions_and_roster_df.drop(columns='Email', inplace=True)

In [35]:
merged_attributions_and_roster_df

,FullName,Username,RosterUUID,TeamID,Team,Year,Role,Task,TaskDescription,Participant
0,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,conceptualization,Was a part of brainstorming for the project id...,Mariel Turkia
1,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,project administration,Organized and supervised the work that the tea...,Mariel Turkia
2,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,entrepreneurship,Worked on the business analysis models,Mariel Turkia
3,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,public engagement,Planned and organized multiple education visit...,Mariel Turkia
4,Mariel Turkia,mariel232,11b7949a-55c0-4952-98a2-675f365d93f8,4831,ABOA-Turku,2023,Student Leader,writing,"Wrote the Education, Collaborations, Implement...",Mariel Turkia
...,...,...,...,...,...,...,...,...,...,...
38816,Aikaterini Eirini Zacharia,kateirzach,f13f60ba-8046-4b95-9dec-b2e3b58dc82d,4983,uniCRETE,2023,Student Leader,public engagement,NaN,Aikaterini Eirini Zacharia
38817,Aikaterini Eirini Zacharia,kateirzach,f13f60ba-8046-4b95-9dec-b2e3b58dc82d,4983,uniCRETE,2023,Student Leader,visualization,NaN,Aikaterini Eirini Zacharia
38818,Aikaterini Eirini Zacharia,kateirzach,f13f60ba-8046-4b95-9dec-b2e3b58dc82d,4983,uniCRETE,2023,Student Leader,writing,NaN,Aikaterini Eirini Zacharia
38819,Sofia Bikaki,sofiabik,ef0788f6-98d2-4bab-b954-0a312de4ec3d,4983,uniCRETE,2023,Student,visualization,NaN,Sofia Bikaki


In [36]:
print(len(merged_attributions_and_roster_df))
print(len(attributions_2023_df))

38821
38812


In [37]:
merged_attributions_and_roster_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38821 entries, 0 to 38820
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   FullName         38821 non-null  object
 1   Username         38821 non-null  object
 2   RosterUUID       38821 non-null  object
 3   TeamID           38821 non-null  int64 
 4   Team             38821 non-null  object
 5   Year             38821 non-null  int64 
 6   Role             38821 non-null  object
 7   Task             38821 non-null  object
 8   TaskDescription  13872 non-null  object
 9   Participant      38130 non-null  object
dtypes: int64(2), object(8)
memory usage: 3.0+ MB


In [38]:
# Print where Participant is null

# Filter for rows where Participant is null
null_participants = merged_attributions_and_roster_df[merged_attributions_and_roster_df['Participant'].isnull()]

null_participants

,FullName,Username,RosterUUID,TeamID,Team,Year,Role,Task,TaskDescription,Participant
110,Mohammad Tarek Mansour,mohammadtarekmansour,211a0306-0a8d-46cb-b781-77fc68701ac5,4586,AFCM-Egypt,2023,Instructor,analysis,NaN,NaN
111,Mohammad Tarek Mansour,mohammadtarekmansour,211a0306-0a8d-46cb-b781-77fc68701ac5,4586,AFCM-Egypt,2023,Instructor,investigation,NaN,NaN
289,Wanyi Wang,wanyiwang,798fc61c-9ed7-493b-98e6-6daa4acf86a6,4808,AIS-China,2023,Student,background research,NaN,NaN
290,Wanyi Wang,wanyiwang,798fc61c-9ed7-493b-98e6-6daa4acf86a6,4808,AIS-China,2023,Student,safety,NaN,NaN
291,Wanyi Wang,wanyiwang,798fc61c-9ed7-493b-98e6-6daa4acf86a6,4808,AIS-China,2023,Student,public engagement,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
38179,Zhang Sanchu,zhangsanchu,52394ac3-8663-4138-ab27-9407845ddc50,4776,ZJUintl-China,2023,Student Leader,public engagement,NaN,NaN
38180,Zhang Sanchu,zhangsanchu,52394ac3-8663-4138-ab27-9407845ddc50,4776,ZJUintl-China,2023,Student Leader,entrepreneurship,NaN,NaN
38181,Zhang Sanchu,zhangsanchu,52394ac3-8663-4138-ab27-9407845ddc50,4776,ZJUintl-China,2023,Student Leader,visualization,NaN,NaN
38182,Zhang Sanchu,zhangsanchu,52394ac3-8663-4138-ab27-9407845ddc50,4776,ZJUintl-China,2023,Student Leader,writing,NaN,NaN


In [39]:
null_participants['FullName'].nunique()

140

In [40]:
print(null_participants[['FullName', 'Username', 'Team', 'Participant']].drop_duplicates(subset='Username'))

                     FullName              Username           Team Participant
110    Mohammad Tarek Mansour  mohammadtarekmansour     AFCM-Egypt         NaN
289                Wanyi Wang             wanyiwang      AIS-China         NaN
313               Yolanda Yau            yolandayau  ANU-Australia         NaN
386                 Kai Hyodo              kaihyodo     ASIJ-Tokyo         NaN
865            Laetitia HOUOT                lhouot  Aix-Marseille         NaN
...                       ...                   ...            ...         ...
37324              Zixin Rong             zixinrong    XJTLU-CHINA         NaN
37453               Jiaqi Xie              28350518      XJU-China         NaN
37507              Yannan You            xy0503e5bd      XMU-China         NaN
38086                Jiyue Hu               jiyuehu     ZJUT-China         NaN
38174            Zhang Sanchu           zhangsanchu  ZJUintl-China         NaN

[140 rows x 4 columns]


There are 140 people (140 null Participant values in the merged column) whose usernames don't match in roster and attributions data (they are usually completely different). We will fuzzy match the remaining names from the merged df and the roster df within one team.

## 3. Fuzzy Matching of Names

Fuzyy matching of roster and attributions names within each team.

In [41]:
# Fuzzy Match FullNames from merged_attributions_and_roster_df with Participant from roster
# Match names within the same team
# Only match if merged_attributions_and_roster_df['Participant'] is null
# Fill the null value in merged_attributions_and_roster_df['Participant'] with the best match from roster_2023_df['Participant']
# If match is not found, keep the original FullName in merged_attributions_and_roster_df['Participant']


def fuzzy_match_names(df1, column1, df2, column2, team_column="Team"):
    """
    Fuzzy matches names from df1[column1] with df2[column2], scoped by team.
    Adds a new column 'Participant' to df1 with the best match from df2[column2].
    
    If no match is found above the threshold, the original value is used.
    Also returns a list of unmatched names.
    
    Parameters:
        df1 (dataframe): First dataframe with names to match, the attributions dataframe.
        column1 (str): Column name in df1 with names to match (like in attributions df "FullName").
        df2 (dataframe): Second dataframe with reference names, the name map df.
        column2 (str): Column name in df2 with names to match against (like in name map df "Participant").
    
    Returns:
        df1 (DataFrame): Original df1 with added 'Participant' column.
        unmatched_df (DataFrame): DataFrame with columns ['Team', 'FullName'] for unmatched entries.
    """
    df1 = df1.copy() # so we don't modify the original df1
    unmatched_rows = []

    if column1 in df1.columns and column2 in df2.columns and team_column in df1.columns and team_column in df2.columns:
        def match_within_team(row):
            if pd.isna(row["Participant"]):  # Only update if Participant is null
                team = row[team_column]
                name = row[column1]

                team_choices = df2[df2[team_column] == team][column2].dropna().unique().tolist()
                result = process.extractOne(name, team_choices, scorer=fuzz.token_sort_ratio, score_cutoff=80)

                if result:
                    return result[0]
                else:
                    unmatched_rows.append({team_column: team, column1: name})
                    return name  # Fallback to original FullName
            return row["Participant"]  # Leave existing value untouched

        df1["Participant"] = df1.apply(match_within_team, axis=1)

    unmatched_df = pd.DataFrame(unmatched_rows)
    return df1, unmatched_df

In [42]:
attributions_2023_df_matched, unmatched_names_df = fuzzy_match_names(
                                                            df1 = merged_attributions_and_roster_df,
                                                            column1="FullName",
                                                            df2=roster_2023_df,
                                                            column2="Participant",
                                                            team_column="Team",
                                                        )

In [43]:
attributions_2023_df_matched.shape

(38821, 10)

In [44]:
unmatched_names_df["FullName"].nunique()

2

In [ ]:
attributions_2023_df_matched["Participant"].isna().sum()

np.int64(0)

In [45]:
unique_unmatched = unmatched_names_df[['FullName', 'Team']].drop_duplicates()

print(unique_unmatched)

               FullName      Team
0           今井 リサ_私の_妻だ   TSBC-SZ
11  Claire Tsitsigianni  Thessaly


Only two people left unmatched but their names and usernames completely differ in both data frames.

In [47]:
# Reorder columns so Participant is second

new_column_order = [
    "FullName",
    "Participant", 
    "RosterUUID",
    "Username",
    "TeamID", 
    "Team", 
    "Year", 
    "Role", 
    "Task", 
    "TaskDescription"
]

attributions_2023_df_matched = attributions_2023_df_matched[new_column_order]

## 4. Hashing Participant Names and Usernames

In [48]:
# Hash FullName in attributions_df with SHA-256

'''
def hash_fullname_column(df, column):
    df_copy = df.copy()
    df_copy[column] = df[column].apply(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())
    return df_copy
'''

def hash_column(df, column):
    df_copy = df.copy()
    df_copy[column] = df_copy[column].apply(
        lambda x: hashlib.sha256(x.strip().lower().encode('utf-8')).hexdigest()
        if isinstance(x, str) else None
    )
    return df_copy

In [49]:
attributions_2023_df_anonymized = hash_column(attributions_2023_df_matched, "Participant")
attributions_2023_df_anonymized = hash_column(attributions_2023_df_anonymized, "Username")


attributions_2023_df_anonymized.head()

,FullName,Participant,RosterUUID,Username,TeamID,Team,Year,Role,Task,TaskDescription
0,Mariel Turkia,1a5551a6dbc4b38b62e580f210bb2aa27b786399ef789e...,11b7949a-55c0-4952-98a2-675f365d93f8,3d38e0990faa2fd0f03b9cbcedf80e22d113090ffba0e5...,4831,ABOA-Turku,2023,Student Leader,conceptualization,Was a part of brainstorming for the project id...
1,Mariel Turkia,1a5551a6dbc4b38b62e580f210bb2aa27b786399ef789e...,11b7949a-55c0-4952-98a2-675f365d93f8,3d38e0990faa2fd0f03b9cbcedf80e22d113090ffba0e5...,4831,ABOA-Turku,2023,Student Leader,project administration,Organized and supervised the work that the tea...
2,Mariel Turkia,1a5551a6dbc4b38b62e580f210bb2aa27b786399ef789e...,11b7949a-55c0-4952-98a2-675f365d93f8,3d38e0990faa2fd0f03b9cbcedf80e22d113090ffba0e5...,4831,ABOA-Turku,2023,Student Leader,entrepreneurship,Worked on the business analysis models
3,Mariel Turkia,1a5551a6dbc4b38b62e580f210bb2aa27b786399ef789e...,11b7949a-55c0-4952-98a2-675f365d93f8,3d38e0990faa2fd0f03b9cbcedf80e22d113090ffba0e5...,4831,ABOA-Turku,2023,Student Leader,public engagement,Planned and organized multiple education visit...
4,Mariel Turkia,1a5551a6dbc4b38b62e580f210bb2aa27b786399ef789e...,11b7949a-55c0-4952-98a2-675f365d93f8,3d38e0990faa2fd0f03b9cbcedf80e22d113090ffba0e5...,4831,ABOA-Turku,2023,Student Leader,writing,"Wrote the Education, Collaborations, Implement..."


In [50]:
attributions_2023_df_matched.head()

,FullName,Participant,RosterUUID,Username,TeamID,Team,Year,Role,Task,TaskDescription
0,Mariel Turkia,Mariel Turkia,11b7949a-55c0-4952-98a2-675f365d93f8,mariel232,4831,ABOA-Turku,2023,Student Leader,conceptualization,Was a part of brainstorming for the project id...
1,Mariel Turkia,Mariel Turkia,11b7949a-55c0-4952-98a2-675f365d93f8,mariel232,4831,ABOA-Turku,2023,Student Leader,project administration,Organized and supervised the work that the tea...
2,Mariel Turkia,Mariel Turkia,11b7949a-55c0-4952-98a2-675f365d93f8,mariel232,4831,ABOA-Turku,2023,Student Leader,entrepreneurship,Worked on the business analysis models
3,Mariel Turkia,Mariel Turkia,11b7949a-55c0-4952-98a2-675f365d93f8,mariel232,4831,ABOA-Turku,2023,Student Leader,public engagement,Planned and organized multiple education visit...
4,Mariel Turkia,Mariel Turkia,11b7949a-55c0-4952-98a2-675f365d93f8,mariel232,4831,ABOA-Turku,2023,Student Leader,writing,"Wrote the Education, Collaborations, Implement..."


In [57]:
# Make matching table for names and hashes

# Drop (Username, Team) duplicates in attributions_2023_df_matched
# Keep columns FullName (renamed to AttributionFormName), Participant (Renamed to Roster2023Name), another copy of Participant but renamed to Roster2023NameAnon,  TeamID, Team, Year
# Hash Roster2023NameAnon

members_matching_table = (
    attributions_2023_df_matched
    .drop_duplicates(subset=["Username", "Team"])
    .rename(columns={
        "FullName": "AttributionFormName",
        "Participant": "Roster2023Name",
    })
)
members_matching_table['Roster2023NameAnon'] = members_matching_table['Roster2023Name']
members_matching_table = members_matching_table[['AttributionFormName', 'Roster2023Name','Roster2023NameAnon', 'RosterUUID', 'TeamID', 'Team', 'Year']] 
members_matching_table = hash_column(members_matching_table, 'Roster2023NameAnon')
members_matching_table.to_csv("../data/igem_ties_surveys/2023/members_matching_table.csv", index=False) 

In [ ]:
# example - check one name
df_example = pd.DataFrame({"Participant": ["Lars Blank"]})
df_example = hash_column(df_example, "Participant")
df_example["Participant"]

0    34e1ec9610e56d588a3ec618ebf3dc2a845d33ca0f147b...
Name: Participant, dtype: object

In [ ]:
attributions_2023_df_anonymized.to_csv("../data/attributions/attributions_anonymized_2023.tsv", sep="\t", index=False)

# file has FullName that is not anonymized and comes from original attribution-form - drop the column later on
# Participant is anonymized and comes from the roster data
# Username is also anonymized

## 5. Merging Survey and Attributions Data 

Merge on hashed names of participants in both dataframes and add a hashed Username column to survey data (and the combined data as well).

In [ ]:
# Check if there are any matches between the hashes (unique matches so one person is counted once)

unique_hashed_fullnames = set(attributions_2023_df_anonymized["Participant"].unique())
unique_participants = set(survey_tasks_df_with_cleaned_task_name["participant"].unique())

# Find intersection
unique_matches = unique_hashed_fullnames & unique_participants

print(f"Number of unique matching hashed names: {len(unique_matches)}")

Number of unique matching hashed names: 449


In [ ]:
survey_tasks_df_with_cleaned_task_name['participant'].nunique()

473

In [ ]:
unmatched_from_survey = unique_participants - unique_hashed_fullnames

print(f"Number of unmatched hashed Participants from survey tasks to FullNames in attributions: {len(unmatched_from_survey)}")
print("Unmatched participant hashes:")

unmatched_from_survey

Number of unmatched hashed Participants from survey tasks to FullNames in attributions: 24
Unmatched participant hashes:


{'08ca0cbaa580d32ebe9f645006fda3ffb575a92824dbf39310f7518016fad08b',
 '0eb9b1f38d710afc5893a5059d4a72628fffef105f6d5e2916b39902daf8bfa9',
 '1455bcf8c71d90e2030ab513398952d039863fa9606b833dc46dcc7591428a24',
 '19a0337425c940ebfd4116c89b3fcd3dcfe894a2342deaf550a23c8b526bfcf7',
 '1c49fab8a067ed513936c7949062d84c3459fa2c188b41f26c9ddb9bd8bf19bd',
 '2c3ad95db6978476ab384ed71ebea42284c8cbbb0b8b037e21ffac8175ce9168',
 '35a854b6aad3d636d50e40b2f2b6c63c0718042b4243018f00061ecaa66e8081',
 '369edee2831f6134b91595a134ead1d975a20e5b251e7ebf513bc81c10e3e222',
 '5785e590b5488ee93901d9c6613af56458c761bba98fafb1748c88fe02ded16f',
 '5acb65af4277cc33fd9e5111c9b04cff37ad7577aaf09f1a0e7088ad79ff21e8',
 '790d8031a0e96ffbfcc14b1fd49c7a9b00f43237618f815b39150029a41884d0',
 '7fa08c843f4e8e86a7bd52cc3750ba9f5c0d0d33f9b1105c737a219cf6510c39',
 '942105ce618cb9c6b7d9d666e2bdbff542596c3fcf563ed7903e672453349be1',
 'a613d6cd42942096684c2e6a1d3a8c75ee4611ccc8cd8def953a0f82df8d730e',
 'b2bd847973677b37e700dd1d5babc570

In [ ]:
unmatched_teams = survey_tasks_df_with_cleaned_task_name[
    survey_tasks_df_with_cleaned_task_name["participant"].isin(unmatched_from_survey)
][["participant", "team"]].drop_duplicates()

print(unmatched_teams)

                                            participant         team
600   19a0337425c940ebfd4116c89b3fcd3dcfe894a2342dea...      Calgary
615   35a854b6aad3d636d50e40b2f2b6c63c0718042b424301...      Calgary
630   a613d6cd42942096684c2e6a1d3a8c75ee4611ccc8cd8d...      Calgary
645   b2bd847973677b37e700dd1d5babc570fa5fbbf92f3487...      Calgary
660   b2e61e583f9c401363d2622c19a9749d2e8b3632c3dc11...      Calgary
675   c85e8eefafd3fcb102bdffcbd7ed1b0d5ee44a5596cfb3...      Calgary
690   e676dbabeed3714a38470c32c9538f3fd26126451338ed...      Calgary
705   fb6332f7021472d9783198abd32b8c30e8d7fc434df7e7...      Calgary
2970  5785e590b5488ee93901d9c6613af56458c761bba98faf...  Montpellier
2985  790d8031a0e96ffbfcc14b1fd49c7a9b00f43237618f81...  Montpellier
3960  2c3ad95db6978476ab384ed71ebea42284c8cbbb0b8b03...     RUBochum
3975  e2ade6d6e503c02a8a266d87e1a7c9dcdfa8d58b488bec...     RUBochum
3990  e3345d4e00c74b98acf210b13961d1381f9c5a0f09dc3c...     RUBochum
4005  f59606e6b868c47ec57785da3b9c

The teams 4944 Calgary, 4741 RUBochum, 4866 UIncheon do not have attributions forms, so they are not found in the attributions_2023_2024.tsv. However, they do have their Wiki pages where they mention attributions and when making an LLM to extract tasks performed for 2022 when the tasks were not structured, these teams from 2023 can be considered as well. A list of teams that did not have an attributions form in 2023 and 2024 can be found in 1_extract_attributions.ipynb.

The team 4697 Montpellier was disqualified in 2023, so it is not included in the attributions_2023_2024.tsv

The two people from 4983 uniCRETE and 4707 UVU-Utah are not mentioned in the attribution form (while they are mentioned in the team's wiki).

## 6. TaskPerformed Column 

Now, we want to combine attributions and survey tasks into a third dataframe that has a binary TaskPerformed column, depending on whether a member has performed a task or not. We will save the final dataframe to the results directory.


In [ ]:
def combine_survey_tasks_with_attributions(survey_df, attributions_df,
                                           s_task_col, a_task_col,
                                           s_participant_col, a_participant_col):
    """
    Combine survey tasks with attributions, adding 'TaskPerformed' and 'Role' columns gathered from attributions.
    
    Parameters:
    - survey_df (DataFrame): dataframe with survey task responses
    - attributions_df (DataFrame): dataframe with team attributions
    - s_task_col (str): column name in survey_df for task names ('Task')
    - a_task_col (str): column name in attributions_df for task names ('Task')
    - s_participant_col (str): column name in survey_df for participant hash ('participant')
    - a_participant_col (str): column name in attributions_df for participant hash ('Participant')

    Returns:
    - df: a copy of survey_df with an additional 'TaskPerformed' column:
        - True: participant-task pair exists in attributions
        - False: participant exists, but task not TaskPerformed
        - NA: participant not found in attributions
    and 'Role' column - the participant's role from attributions_df, matched by participant name.
    """
    df = survey_df.copy()
    attributions_copy = attributions_df.copy()

    # Create merge key for exact participant-task match
    df['_merge_key'] = df[s_participant_col] + '||' + df[s_task_col]
    attributions_copy['_merge_key'] = attributions_copy[a_participant_col] + '||' + attributions_copy[a_task_col]

    # Assign TaskPerformed based on composite key
    df['TaskPerformed'] = df['_merge_key'].isin(attributions_copy['_merge_key']).astype('Int64')

    # Set TaskPerformed = pd.NA where participant is not found in attributions
    unmatched_participants = ~df[s_participant_col].isin(attributions_copy[a_participant_col])
    df.loc[unmatched_participants, 'TaskPerformed'] = pd.NA

    # Add Role by merging on participant only (not task), avoiding column collision
    roles = attributions_copy[[a_participant_col, 'Role']].drop_duplicates(subset=[a_participant_col])
    roles = roles.rename(columns={'Role': '_Role'})  # temporary name to avoid conflict
    df = df.merge(roles, left_on=s_participant_col, right_on=a_participant_col, how='left')
    df.rename(columns={'_Role': 'Role'}, inplace=True)

    # Cleanup
    df.drop(columns=['_merge_key', a_participant_col], inplace=True)

    return df


In [ ]:
combined_df = combine_survey_tasks_with_attributions(
    survey_df=survey_tasks_df_with_cleaned_task_name,
    attributions_df=attributions_2023_df_anonymized,
    s_task_col='Task',
    a_task_col='Task',
    s_participant_col='participant',
    a_participant_col='Participant'
)

In [ ]:
# Rename the participant column to TeamMember 
combined_df.rename(columns={'participant': 'TeamMember'}, inplace=True)

In [ ]:
combined_df.to_csv("../data/survey_tasks_and_attributions/2023_survey_tasks_and_attributions.tsv", sep="\t", index=False)

In [ ]:
combined_df.head()

,team,TeamMember,Task,Certainty,Experience,Preference,TaskPerformed,Role
0,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,analysis,Somewhat unsure,Between one and two years,Neutral,0,Student Leader
1,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,background research,Somewhat sure,Between one and two years,Neutral,1,Student Leader
2,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,conceptualization,Neutral,Less than one year,Neutral,1,Student Leader
3,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,data curation,Somewhat unsure,Less than one year,Somewhat disagree,0,Student Leader
4,ABOA-Turku,172d6820fb71f042f13fb7e22ffd437221944e3757ff20...,entrepreneurship,Neutral,Less than one year,Somewhat disagree,0,Student Leader


In [ ]:
survey_tasks_df_with_cleaned_task_name.shape

(7095, 6)

In [ ]:
# Total number of rows
total_rows = len(combined_df)

# Number of rows where TaskPerformed == 0
task_not_performed = (combined_df['TaskPerformed'] == 0).sum()

# Number of rows where TaskPerformed == 1
task_performed = (combined_df['TaskPerformed'] == 1).sum()

# Number of rows where TaskPerformed is null 
task_performed_null = combined_df['TaskPerformed'].isna().sum()

print(f"Total rows in combined dataframe: {total_rows}")
print(f"TaskPerformed = 0: {task_not_performed}")
print(f"TaskPerformed = 1: {task_performed}")
print(f"TaskPerformed = NA: {task_performed_null}")

Total rows in combined dataframe: 7095
TaskPerformed = 0: 3921
TaskPerformed = 1: 2814
TaskPerformed = NA: 360


In [ ]:
# Cleanup of the column names and values - Run with final results

# Capitalize the first letter of each Task entry
combined_df['Task'] = combined_df['Task'].str.title()

# Replace dashes with spaces in Role, then capitalize first letter
combined_df['Role'] = (
    combined_df['Role']
    .str.replace('-', ' ')
    .str.title()
)

combined_df['Role'] = (
    combined_df['Role']
    .replace({
        'Student': 'Student Member',
        'Primary Pi': 'Primary PI',
        'Secondary Pi': 'Secondary PI'
    })
)

# Rename the “team” column to “Team”
combined_df.rename(columns={'team': 'Team'}, inplace=True)

# Reorder columns
new_column_order = [
    "Team",
    "TeamMember",  
    "Role", 
    "Task", 
    "Certainty",
    "Experience",
    "Preference",
    "TaskPerformed"
]
combined_df = combined_df[new_column_order]

# Save as TSV
combined_df.to_csv("../results/survey_tasks_and_attributions/2023_survey_tasks_and_attributions_combined.tsv", sep="\t", index=False)